In [0]:
import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql('select current_timestamp()').collect()[0][0]

dbutils.widgets.text("table_metadata","{'table_id': '1','table_name': 'customers','source_system': 'sqlserver','source_schema':'banking',    'source_table':'customers','source_path':'','bronze_schema': 'bronze','silver_schema': 'silver','active_flag':'true', 'load_order':'1','created_at':'2026-03-30 12:50:31.872507'}")

dbutils.widgets.text("table_parameters","{'load_type': 'MERGE', 'primary_key': 'customer_id', 'watermark_column': 'updated_at'}")

dbutils.widgets.text("run_id","85503500349963")

run_id = dbutils.widgets.get("run_id")


table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'",'"'))
table_parameters = json.loads(dbutils.widgets.get("table_parameters").replace("'",'"'))

print("Table Metadata:", table_metadata)
print("Table Parameters:", table_parameters)
print(f"Run ID: {run_id}")

In [0]:
table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"]
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]

load_type = table_parameters.get("load_type")
watermark_column = table_parameters.get("watermark_column")

bronze_table_fqn = f"banking.{bronze_schema}.{table_name}"
print(f"Target Bronze Table: {bronze_table_fqn}")

In [0]:
entry_exits = spark.sql(f"""
                        select 1
                        from banking.metadata.pipeline_runs
                        where run_id = {run_id} and table_id = {table_id}
                        """).count() >0


if entry_exits:
    spark.sql(f"""
              update banking.metadata.pipeline_runs
              set
              layer ='Silver',
              start_time = timestamp('{start_time}'),
              end_time = NULL,
              status = "INPROGRESS",
              num_of_records = NULL,
              error_message = NULL
              where run_id = {run_id} and table_id={table_id}
              """)
else:
    spark.sql(f"""
              INSERT INTO banking.metadata.pipeline_runs
              VALUES(
                  {run_id},
                  {table_id},
                  'Silver',
                  Timestamp('{start_time}'),
                  NULL,
                  'INPROGRESS',
                  NULL,
                  NULL
              )
              """)

In [0]:
last_watermark = None
if load_type in ['APPEND','MERGE'] and watermark_column:
    watermark_df = spark.sql(f"""
                             select last_watermark_value
                             from banking.metadata.table_watermarks
                             where table_id = {table_id}
                             """)
    
    if watermark_df.count()>0:
        last_watermark = watermark_df.first()['last_watermark_value']

print("Last Watermark:", last_watermark)

In [0]:
%sql
create schema if not exists banking.bronze

In [0]:
try:
    if source_system == "sqlserver":
        secret_json = dbutils.secrets.get(
            scope="banking-scope",
            key="sqlserver-connection-json"
        )

        config = json.loads(secret_json)

        jdbc_url = f"jdbc:sqlserver://{config['host']}:{config['port']};database={config['database']}"
        print(jdbc_url)

        jdbc_properties = {
            "user" : config["user"],
            "password": config["password"],
            "driver": config["driver"]
        }

        if load_type in ['APPEND','MERGE'] and last_watermark:
            query = f"""
            (select * from {source_schema}.{source_table}
            where {watermark_column} > '{last_watermark}') as src
            """
        else:
            query = f"(select * from {source_schema}.{source_table}) as src"


        source_df = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties = jdbc_properties
        )

    elif source_system == 'blob':
                source_df = (
                    spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "csv")
                    .option(
                        "cloudFiles.schemaLocation",
                        f"/Volumes/banking/source/volume/_schema/{table_name}"
                    )
                    .option("header", "true")
                )
    else:
            raise ValueError("Unsupported source_system")


    source_df = source_df.withColumn("insert_timestamp",current_timestamp())


    if source_system == 'blob':
            (
                source_df.writeStream
                .format("delta")
                .option(
                    "checkpointLocation",
                    f"Volumnes/banking/source/volume/_checkpoint/{table_name}"
                )
                .outputMode("append")
                .trigger(availableNow=True)
                .toTable(bronze_table_fqn)
            )

            records_read = None
    else:

            (
                source_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(bronze_table_fqn)
            )

            records_read = source_df.count()


    print("Source-> Bronze Load Completed Successfully.")
    print("wartermark will be updated after Silver load.")

except Exception as e:
    end_time = spark.sql("select current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
              update banking.metadata.pipeline_runs
              set
              end_time = timestamp('{end_time}'),
              status ='FAILED',
              error_message= {'NULL' if not error_message else "'" + error_message.replace("'","") +"'"}
              where table_id = {table_id} and run_id = {run_id}
              """)
    raise